### Step-1:Load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\HP\OneDrive\Desktop\projects\StockSense360")
DATA = ROOT / "02_data_processed"
OUT = ROOT / "08_outputs"
OUT.mkdir(exist_ok=True)

fact = pd.read_csv(DATA / "fact_sales.csv", parse_dates=["date"])
print("fact shape:", fact.shape)
fact.head()

fact shape: (956500, 8)


,date,wm_yr_wk,store_id,state_id,item_id,dept_id,cat_id,units_sold
0,2011-01-29,11101,CA_1,CA,FOODS_3_180,FOODS_3,FOODS,0
1,2011-01-29,11101,CA_3,CA,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,2
2,2011-01-29,11101,CA_3,CA,FOODS_3_409,FOODS_3,FOODS,0
3,2011-01-29,11101,CA_2,CA,FOODS_1_097,FOODS_1,FOODS,0
4,2011-01-29,11101,TX_2,TX,HOBBIES_1_272,HOBBIES_1,HOBBIES,0


### Step-2:Choose store + 50 items

In [2]:
store = "CA_1"
items = fact[fact["store_id"] == store]["item_id"].unique()[:50]

df = fact[(fact["store_id"] == store) & (fact["item_id"].isin(items))].copy()
print("store:", store, "items:", len(items), "rows:", df.shape[0])
df.head()

store: CA_1 items: 41 rows: 78433


,date,wm_yr_wk,store_id,state_id,item_id,dept_id,cat_id,units_sold
0,2011-01-29,11101,CA_1,CA,FOODS_3_180,FOODS_3,FOODS,0
8,2011-01-29,11101,CA_1,CA,HOUSEHOLD_1_537,HOUSEHOLD_1,HOUSEHOLD,3
11,2011-01-29,11101,CA_1,CA,FOODS_3_096,FOODS_3,FOODS,0
21,2011-01-29,11101,CA_1,CA,FOODS_2_382,FOODS_2,FOODS,2
28,2011-01-29,11101,CA_1,CA,HOUSEHOLD_1_202,HOUSEHOLD_1,HOUSEHOLD,0


### Step-3:Daily time series per item

In [3]:
daily = (df.groupby(["item_id", "date"])["units_sold"]
           .sum()
           .reset_index()
           .sort_values(["item_id", "date"]))

print("daily shape:", daily.shape)
daily.head()

daily shape: (78433, 3)


,item_id,date,units_sold
0,FOODS_1_005,2011-01-29,3
1,FOODS_1_005,2011-01-30,9
2,FOODS_1_005,2011-01-31,3
3,FOODS_1_005,2011-02-01,3
4,FOODS_1_005,2011-02-02,0


### Step-4:Forecast 14 days (Seasonal Naive = repeat last 7 days)

In [4]:
horizon = 14
forecasts = []

for item in daily["item_id"].unique():
    s = (daily[daily["item_id"] == item]
         .set_index("date")["units_sold"]
         .asfreq("D")
         .fillna(0))

    last_date = s.index.max()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq="D")

    last7 = s.tail(7).values
    pred = np.resize(last7, horizon)

    temp = pd.DataFrame({
        "store_id": store,
        "item_id": item,
        "date": future_dates,
        "forecast_units": pred
    })
    forecasts.append(temp)

forecast_df = pd.concat(forecasts, ignore_index=True)

print("forecast_df shape:", forecast_df.shape)
forecast_df.head(10)

forecast_df shape: (574, 4)


,store_id,item_id,date,forecast_units
0,CA_1,FOODS_1_005,2016-04-25,2
1,CA_1,FOODS_1_005,2016-04-26,0
2,CA_1,FOODS_1_005,2016-04-27,2
3,CA_1,FOODS_1_005,2016-04-28,2
4,CA_1,FOODS_1_005,2016-04-29,1
5,CA_1,FOODS_1_005,2016-04-30,4
6,CA_1,FOODS_1_005,2016-05-01,1
7,CA_1,FOODS_1_005,2016-05-02,2
8,CA_1,FOODS_1_005,2016-05-03,0
9,CA_1,FOODS_1_005,2016-05-04,2


### Step-5:Save forecast file

In [5]:
forecast_df.to_csv(OUT / "forecast_14days_store_CA1.csv", index=False)
print("✅ Saved:", OUT / "forecast_14days_store_CA1.csv")

✅ Saved: C:\Users\HP\OneDrive\Desktop\projects\StockSense360\08_outputs\forecast_14days_store_CA1.csv
